# aura-learn — 7B QLoRA SFT on Kaggle T4

Runs the same `aura_learn.train.sft` module used locally, on the 7B base model.
Kaggle gives a free T4 (16 GB); 7B QLoRA fits. Session limit is 12 h.

Setup: upload `corrections.jsonl` (from `aura-learn export --out corrections.jsonl`)
as a Kaggle dataset.

In [ ]:
# 1. Install (CUDA torch is preinstalled on Kaggle; add the rest)
!pip install -q transformers peft trl datasets pyyaml
!pip install -q git+https://github.com/DusanCar-sudo/aura-learn.git

In [ ]:
# 2. Point at your correction data (upload as a Kaggle dataset)
DATA = "/kaggle/input/corrections/corrections.jsonl"

In [ ]:
# 3. Train — identical code path to the local CPU smoke test
from aura_learn.train.sft import run_sft
from pathlib import Path

adapter = run_sft(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    data_path=Path(DATA),
    output_dir=Path("/kaggle/working/adapter"),
    max_steps=300,
    learning_rate=5e-5,
    lora_r=16,
    lora_alpha=32,
)
print("adapter saved to", adapter)

In [ ]:
# 4. Sanity check: the adapter loads back onto the base model
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, "/kaggle/working/adapter")
print("adapter loads OK")

5. Save version → download `adapter/`. Guard check (personal + forgetting benchmarks) happens back home in Phase 2 — **no adapter reaches aura-code without passing the guard.**